In [1]:
import ee
import geemap
import pandas as pd

# 1. Inisialisasi Google Earth Engine
ee.Initialize(project='flowing-tea-463503-c6')

In [ ]:
# 2. Muat Shapefile Lokal sebagai AOI
# Ubah path sesuai lokasi file .shp Anda di komputer
shp_path = "D:/jambi/HD/Data Cfes/Pengasih_baru_HD.shp"
aoi_ee = geemap.shp_to_ee(shp_path)

In [2]:
# Panggil Asset SHP yang sudah di-upload ke GEE
# Contoh ID: projects/flowing-tea-463503-c6/assets/nama_aoi_anda
aoi_ee = ee.FeatureCollection("projects/flowing-tea-463503-c6/assets/Manggrove_AP_diss")



In [3]:
# 3. Panggil Dataset JRC TMF Annual Changes
# -------------------------------------------------------------------------
# SESUAI DOKUMENTASI RESMI JRC TMF GEE TUTORIAL
# Asset ID: "projects/JRC/TMF/v1_subannual_annual_changes"
# Tipe: Single ee.Image
# -------------------------------------------------------------------------
# -------------------------------------------------------------------------
# SOLUSI PASTI: Panggil sebagai ImageCollection dari Catalog Publik GEE
# -------------------------------------------------------------------------

try:
    # 1. Panggil ImageCollection Annual Changes JRC TMF
    tmf_col = ee.ImageCollection("projects/JRC/TMF/v1_2025/AnnualChanges")
    # Mosaic untuk menggabungkan potongan tile spasial global menjadi 1 citra
    tmf_image = tmf_col.mosaic()
    # Tes verifikasi akses
    _ = tmf_image.bandNames().getInfo()
    print("Berhasil memuat JRC TMF versi v1_2025")
except Exception as e:
    print("Versi v1_2023 tidak merespons, beralih ke v1_2022...")
    tmf_col = ee.ImageCollection("projects/JRC/TMF/v1_2022/AnnualChanges")
    tmf_image = tmf_col.mosaic()

# 2. Ambil daftar band yang tersedia
available_bands = tmf_image.bandNames().getInfo()

# 3. Pilih band Dec2015
tmf_2015 = tmf_image.select('Dec2015')

# 4. Pilih band tahun 'Dec' paling baru yang tersedia (misal: Dec2023 atau Dec2022)
latest_band = [b for b in available_bands if b.startswith('Dec')][-1]
tmf_latest = tmf_image.select(latest_band)

print(f"Perbandingan tutupan lahan: Dec2015 vs {latest_band}")

# 5. Gabungkan nilai piksel: (Kelas_2015 * 10) + Kelas_Tujuan
change_image = tmf_2015.multiply(10).add(tmf_latest).rename('transition')

Berhasil memuat JRC TMF versi v1_2025
Perbandingan tutupan lahan: Dec2015 vs Dec2025


In [4]:
# 4. Fungsi Pemetaan Aturan Transisi
def map_transition(from_class, to_class):
    # Kelas 1 (Undisturbed Forest)
    if from_class == 1:
        if to_class == 1: return 'Remain Forest'
        elif to_class in [2, 4]: return 'Degraded Forest'
        elif to_class in [3, 5, 6]: return 'Deforestation'
        
    # Kelas 2 (Degraded Forest)
    elif from_class == 2:
        if to_class == 1: return 'Forest Enhancement'
        elif to_class in [2, 4]: return 'Remain Forest'
        elif to_class in [3, 5, 6]: return 'Deforestation'
        
    # Kelas 3 (Deforested Land)
    elif from_class == 3:
        if to_class in [1, 2, 4]: return 'Reforestation'
        elif to_class in [3, 5, 6]: return 'Remain Non Forest'
        
    # Kelas 4 (Forest Regrowth)
    elif from_class == 4:
        if to_class in [1, 2]: return 'Forest Enhancement'
        elif to_class == 4: return 'Remain Forest'
        elif to_class in [3, 5, 6]: return 'Deforestation'
        
    # Kelas 5 (Water) & Kelas 6 (Other Land Cover)
    elif from_class in [5, 6]:
        if to_class in [1, 2, 4]: return 'Reforestation'
        elif to_class in [3, 5, 6]: return 'Remain Non Forest'
        
    return 'Unknown'

In [5]:
# 5. Hitung Luas Per Piksel dalam Hektar (ha)
area_image = ee.Image.pixelArea().divide(10000).addBands(change_image)

# Ekstraksi statistik area per kode transisi
stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='transition'
    ),
    geometry=aoi_ee.geometry(),
    scale=30,
    maxPixels=1e13
)

In [6]:
stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='transition'
    ),
    geometry=aoi_ee.geometry(),
    scale=60,  # <-- Ubah scale dari 30 menjadi 60 (atau 100 jika masih memori penuh)
    maxPixels=1e13,
    bestEffort=True  # <-- Tambahkan parameter ini agar GEE otomatis menyesuaikan scale jika memori mepet
)

In [7]:
# 6. Olah Hasil ke Pandas DataFrame
raw_data = stats.get('groups').getInfo()
df = pd.DataFrame(raw_data)

# Parsing nilai transisi kembali ke kelas asal dan kelas tujuan
df['from_class'] = df['transition'].apply(lambda x: int(x // 10))
df['to_class'] = df['transition'].apply(lambda x: int(x % 10))
df.rename(columns={'sum': 'area_ha'}, inplace=True)

# Aplikasikan fungsi matriks transisi kustom
df['category'] = df.apply(lambda row: map_transition(row['from_class'], row['to_class']), axis=1)

# Filter nilai di luar kelas 1-6 jika ada
df_filtered = df[(df['from_class'] >= 1) & (df['from_class'] <= 6) & 
                 (df['to_class'] >= 1) & (df['to_class'] <= 6)]

EEException: User memory limit exceeded.

In [ ]:
# 7. Ringkasan Total Luas per Kategori Perubahan
summary_table = df_filtered.groupby('category')['area_ha'].sum().reset_index()
summary_table['percentage (%)'] = (summary_table['area_ha'] / summary_table['area_ha'].sum()) * 100

print("=== RINGKASAN PERUBAHAN TUTUPAN LAHAN ===")
print(summary_table.sort_values(by='area_ha', ascending=False).to_string(index=False))

# Optional: Simpan ringkasan ke CSV
summary_table.to_csv("ringkasan_perubahan_tmf.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------------------
# A. EKSPOR KE FILE CSV
# -------------------------------------------------------------------------

# 1. Ringkasan Total Luas per Kategori
summary_table = df_filtered.groupby('category')['area_ha'].sum().reset_index()
summary_table['percentage (%)'] = (summary_table['area_ha'] / summary_table['area_ha'].sum()) * 100
summary_table = summary_table.sort_values(by='area_ha', ascending=False)

summary_table.to_csv("ringkasan_perubahan_tutupan_lahan_tmf.csv", index=False)

# 2. Matriks Transisi Detail (From Class vs To Class)
pivot_matrix = df_filtered.pivot_table(index='from_class', columns='to_class', values='area_ha', aggfunc='sum', fill_value=0)
pivot_matrix.to_csv("matriks_detail_transisi_tmf.csv")

print("File CSV berhasil disimpan!")

# -------------------------------------------------------------------------
# B. VISUALISASI DIAGRAM BATANG (STATISTIK PERUBAHAN)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 6))
palette = {
    'Remain Forest': '#2e7d32',        # Hijau Tua
    'Degraded Forest': '#fbc02d',      # Kuning
    'Deforestation': '#d32f2f',       # Merah
    'Forest Enhancement': '#00796b',   # Hijau Toska
    'Reforestation': '#388e3c',       # Hijau Muda
    'Remain Non Forest': '#757575'     # Abu-abu
}

sns.barplot(
    data=summary_table, 
    x='area_ha', 
    y='category', 
    palette=palette
)

plt.title('Ringkasan Perubahan Tutupan Lahan TMF (2015 - Terbaru)', fontsize=14, fontweight='bold')
plt.xlabel('Luas (Hektar)', fontsize=12)
plt.ylabel('Kategori Perubahan', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("grafik_perubahan_tmf.png", dpi=300)
plt.show()

In [ ]:
import geemap

# -------------------------------------------------------------------------
# C. REKLASIFIKASI RASTER DENGAN ATURAN DARI USER
# -------------------------------------------------------------------------
# Kelas Baru:
# 1: Remain Forest | 2: Degraded Forest | 3: Deforestation
# 4: Forest Enhancement | 5: Reforestation | 6: Remain Non Forest

from_values = [
    11, 12, 13, 14, 15, 16,
    21, 22, 23, 24, 25, 26,
    31, 32, 33, 34, 35, 36,
    41, 42, 43, 44, 45, 46,
    51, 52, 53, 54, 55, 56,
    61, 62, 63, 64, 65, 66
]

to_values = [
    1, 2, 3, 2, 3, 3,  # Dari Kelas 1
    4, 1, 3, 1, 3, 3,  # Dari Kelas 2
    5, 5, 6, 5, 6, 6,  # Dari Kelas 3
    4, 4, 3, 1, 3, 3,  # Dari Kelas 4
    5, 5, 6, 5, 6, 6,  # Dari Kelas 5
    5, 5, 6, 5, 6, 6   # Dari Kelas 6
]

# Reklasifikasi citra transisi
reclassified_image = change_image.remap(from_values, to_values).rename('classified_change')

# -------------------------------------------------------------------------
# D. TAMPILKAN PETA INTERAKTIF
# -------------------------------------------------------------------------
Map = geemap.Map()
Map.centerObject(aoi_ee, 10)

# Visualisasi parameter warna
vis_params = {
    'min': 1,
    'max': 6,
    'palette': ['#2e7d32', '#fbc02d', '#d32f2f', '#00796b', '#388e3c', '#757575']
}

# Tambahkan layer AOI & Peta Perubahan
Map.addLayer(aoi_ee, {}, 'Batas AOI (SHP)')
Map.addLayer(reclassified_image.clip(aoi_ee), vis_params, 'Peta Perubahan Tutupan Lahan TMF')

# Tambahkan legenda
legend_keys = [
    '1. Remain Forest', 
    '2. Degraded Forest', 
    '3. Deforestation', 
    '4. Forest Enhancement', 
    '5. Reforestation', 
    '6. Remain Non Forest'
]
legend_colors = ['#2e7d32', '#fbc02d', '#d32f2f', '#00796b', '#388e3c', '#757575']

Map.add_legend(title="Kategori Perubahan TMF", keys=legend_keys, colors=legend_colors)

Map